In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn

In [2]:
df = pd.read_csv(r"fmnist_small.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [ ]:
x = df.iloc[:, 1:]
y = df.iloc[:,0]

In [4]:
x

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,0,0,0,0,0,0,0,0,0,1,...,69,12,0,0,0,0,0,0,0,0
5996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5997,0,0,0,0,0,0,0,0,0,0,...,39,47,2,0,0,29,0,0,0,0
5998,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [5]:
y

0       9
1       7
2       0
3       8
4       8
       ..
5995    1
5996    5
5997    8
5998    4
5999    8
Name: label, Length: 6000, dtype: int64

In [6]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [7]:
x_train = x_train/255.0
x_test = x_test/255.0

In [8]:
class CustomDataset(Dataset):

    def __init__(self, features, labels):
        self.features = torch.tensor(features.values, dtype=torch.float32)
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [9]:
train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

In [18]:
class MyNN(nn.Module):

    def __init__(self, input_dim, output_dim, num_hidden_layer, neuron_per_layer, dropout_rate):
        super().__init__()

        layers = []

        for i in range(num_hidden_layer):
            layers.append(nn.Linear(input_dim, neuron_per_layer))
            layers.append(nn.BatchNorm1d(neuron_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            
            input_dim = neuron_per_layer
        layers.append(nn.Linear(input_dim, output_dim))

        self.model = nn.Sequential(*layers)    

    def forward(self, x):
        return self.model(x)    

In [19]:
device = torch.device('cuda' if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
def object(trial):
    num_hidden_layer = trial.suggest_int('num_hidden_layer', 1, 5)
    neuron_per_layer = trial.suggest_int('neuron_per_layer', 8, 128, step=8)
    epoch = trial.suggest_int('epoch', 0, 100, step=10)
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    dropout_rate = trial.suggest_float('dropout', 0.1, 0.5)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'RMSProp'])
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    #model init
    input_dim = 784
    output_dim = 10

    model = MyNN(input_dim, output_dim, num_hidden_layer, neuron_per_layer, dropout_rate)
    model = model.to(device)



    # Optimizer & Loss function
    loss_function = nn.CrossEntropyLoss()
    if optimizer_name=='SGD':
        optimizer = torch.optim.SGD(model.parameters(), lr, weight_decay=weight_decay)
    elif optimizer_name=='Adam':
        optimizer = torch.optim.Adam(model.parameters(), lr, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.RMSprop(model.parameters(), lr, weight_decay=weight_decay)
    

    # Training Loop
    for epoch in range(epoch):
        total_loss = 0
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            y_pred = model(features)

            loss = loss_function(y_pred, labels)

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

    # Evaluation
    model.eval()

    total = 0
    correct = 0


    with torch.no_grad():
        for features, labels in test_loader:
            features, labels = features.to(device), labels.to(device)

            y_pred = model(features)
            _, predicted = torch.max(y_pred, 1)
            total+=labels.size(0)
            correct+=(predicted==labels).sum().item()
            
    accuracy = correct/total
    return accuracy    

In [30]:
import optuna

study = optuna.create_study(direction="maximize")

[I 2026-05-07 16:55:04,638] A new study created in memory with name: no-name-3648d170-b7cf-4540-ab26-11df2bbc4910


In [31]:
study.optimize(object, n_trials=5)

[I 2026-05-07 16:55:24,525] Trial 0 finished with value: 0.8375 and parameters: {'num_hidden_layer': 1, 'neuron_per_layer': 56, 'epoch': 50, 'lr': 0.003690074269890051, 'dropout': 0.38748099317808693, 'batch_size': 32, 'optimizer': 'Adam', 'weight_decay': 4.6605766547876984e-05}. Best is trial 0 with value: 0.8375.
[I 2026-05-07 16:55:30,522] Trial 1 finished with value: 0.44083333333333335 and parameters: {'num_hidden_layer': 1, 'neuron_per_layer': 56, 'epoch': 40, 'lr': 4.8738936299568245e-05, 'dropout': 0.43333847439456774, 'batch_size': 128, 'optimizer': 'SGD', 'weight_decay': 0.0003403368376925139}. Best is trial 0 with value: 0.8375.
[I 2026-05-07 16:55:46,901] Trial 2 finished with value: 0.83 and parameters: {'num_hidden_layer': 1, 'neuron_per_layer': 48, 'epoch': 50, 'lr': 0.004011919761243438, 'dropout': 0.4329278693123253, 'batch_size': 32, 'optimizer': 'RMSProp', 'weight_decay': 1.1257039561662064e-05}. Best is trial 0 with value: 0.8375.
[I 2026-05-07 16:56:05,372] Trial 3

In [32]:
study.best_trial

FrozenTrial(number=0, state=<TrialState.COMPLETE: 1>, values=[0.8375], datetime_start=datetime.datetime(2026, 5, 7, 16, 55, 5, 26596), datetime_complete=datetime.datetime(2026, 5, 7, 16, 55, 24, 525426), params={'num_hidden_layer': 1, 'neuron_per_layer': 56, 'epoch': 50, 'lr': 0.003690074269890051, 'dropout': 0.38748099317808693, 'batch_size': 32, 'optimizer': 'Adam', 'weight_decay': 4.6605766547876984e-05}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'num_hidden_layer': IntDistribution(high=5, log=False, low=1, step=1), 'neuron_per_layer': IntDistribution(high=128, log=False, low=8, step=8), 'epoch': IntDistribution(high=100, log=False, low=0, step=10), 'lr': FloatDistribution(high=0.1, log=True, low=1e-05, step=None), 'dropout': FloatDistribution(high=0.5, log=False, low=0.1, step=None), 'batch_size': CategoricalDistribution(choices=(32, 64, 128)), 'optimizer': CategoricalDistribution(choices=('Adam', 'SGD', 'RMSProp')), 'weight_decay': FloatDistribution(hi

In [33]:
study.best_params

{'num_hidden_layer': 1,
 'neuron_per_layer': 56,
 'epoch': 50,
 'lr': 0.003690074269890051,
 'dropout': 0.38748099317808693,
 'batch_size': 32,
 'optimizer': 'Adam',
 'weight_decay': 4.6605766547876984e-05}

In [34]:
study.best_value

0.8375